# Module 6: Prompting Techniques

## Learning Objectives
By the end of this module, you will be able to:
- Understand the principles of effective prompt engineering
- Apply zero-shot, few-shot, and chain-of-thought prompting
- Use structured prompts for consistent outputs
- Debug and iterate on prompts for better results

---

## 1. What is Prompt Engineering?

**Prompt Engineering** is the art and science of crafting inputs to LLMs to get desired outputs.

### Why It Matters

```
Same Model + Different Prompts = Very Different Outputs
```

Unlike traditional programming where you write explicit code, with LLMs you **communicate intent through language**.

### The Prompt Engineering Mindset

| Traditional Programming | Prompt Engineering |
|------------------------|--------------------|
| Write explicit algorithms | Describe desired behavior |
| Debug code | Iterate on prompts |
| Compile errors are precise | LLM failures are subtle |
| One correct implementation | Many valid approaches |

---

## 2. Anatomy of a Good Prompt

### Key Components

```
┌────────────────────────────────────────────────────────────────┐
│  ROLE/PERSONA (optional)                                       │
│  "You are an expert Python developer..."                       │
├────────────────────────────────────────────────────────────────┤
│  CONTEXT                                                       │
│  Background information the model needs                        │
├────────────────────────────────────────────────────────────────┤
│  TASK/INSTRUCTION                                              │
│  What you want the model to do                                 │
├────────────────────────────────────────────────────────────────┤
│  FORMAT (optional)                                             │
│  How you want the output structured                            │
├────────────────────────────────────────────────────────────────┤
│  EXAMPLES (optional)                                           │
│  Demonstrations of desired behavior                            │
└────────────────────────────────────────────────────────────────┘
```

### Best Practices

1. **Be Specific**: Vague prompts get vague answers
2. **Provide Context**: Include relevant background
3. **Specify Format**: Tell the model how to structure output
4. **Use Delimiters**: Separate sections with `###`, `---`, or XML tags
5. **Iterate**: Refine based on results

In [ ]:
# Setup: Install required packages
!pip install -q openai python-dotenv

In [ ]:
import os
from openai import OpenAI

# Option 1: Set your API key directly (for Colab)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# Option 2: Load from .env file (for local development)
from dotenv import load_dotenv
load_dotenv()

# Initialize client
client = OpenAI()

def chat(prompt, model="gpt-4o-mini", temperature=0.7):
    """Simple helper function to call OpenAI API"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

print("✅ OpenAI client ready!")

In [ ]:
# Example: Bad vs Good Prompt

# ❌ Bad prompt - vague, no structure
bad_prompt = "Tell me about Python"

# ✅ Good prompt - specific, structured, clear format
good_prompt = """
Provide a brief overview of Python programming for a beginner.

Include:
1. What Python is (1 sentence)
2. Three main use cases
3. One simple code example

Keep the total response under 150 words.
"""

print("💬 Good Prompt Response:\n")
print(chat(good_prompt))

---

## 3. Zero-Shot Prompting

**Zero-shot** means asking the model to perform a task **without any examples** - relying purely on its pre-trained knowledge.

### When to Use
- Simple, well-defined tasks
- Standard formats the model has seen during training
- Quick prototyping

### Examples

In [ ]:
# Zero-shot: Classification
zero_shot_classification = """
Classify the following customer review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The product arrived on time but the packaging was damaged. 
The item itself works fine though."

Classification:
"""

print("Classification result:")
print(chat(zero_shot_classification, temperature=0))

In [ ]:
# Zero-shot: Summarization
zero_shot_summary = """
Summarize the following text in exactly 2 sentences:

---
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing algorithms that can access data and use it to learn 
for themselves. The process begins with observations or data, such as examples, 
direct experience, or instruction, to look for patterns in data and make better 
decisions in the future based on the examples provided.
---

Summary:
"""

print("Summary:")
print(chat(zero_shot_summary, temperature=0))

In [ ]:
# Zero-shot: Code Generation
zero_shot_code = """
Write a Python function that:
1. Takes a list of numbers as input
2. Returns a dictionary with 'min', 'max', and 'average' keys
3. Include a docstring explaining the function

Only output the code, no explanations.
"""

print("Generated code:")
print(chat(zero_shot_code, temperature=0))

---

## 4. Few-Shot Prompting

**Few-shot** prompting provides **examples** of the desired input-output pattern. The model learns the pattern from examples and applies it to new inputs.

### When to Use
- Custom formats not standard in training data
- Specific output styles
- Domain-specific conventions
- Complex classification with edge cases

In [ ]:
# Few-shot: Custom sentiment labeling with explanations
few_shot_sentiment = """
Classify the sentiment and provide a brief reason.

Example 1:
Text: "This laptop exceeds all my expectations!"
Sentiment: POSITIVE
Reason: Strong positive language ("exceeds expectations")

Example 2:
Text: "Decent product for the price point."
Sentiment: NEUTRAL
Reason: Balanced assessment, neither enthusiastic nor critical

Example 3:
Text: "Complete waste of money, broke after one week."
Sentiment: NEGATIVE
Reason: Strong negative language, product failure mentioned

Now classify:
Text: "It does what it's supposed to do, nothing more nothing less."
Sentiment:
Reason:
"""

print("Few-shot classification:")
print(chat(few_shot_sentiment, temperature=0))

In [ ]:
# Few-shot: Custom data extraction format
few_shot_extraction = """
Extract structured information from job postings.

Example 1:
Input: "Senior Software Engineer at Google, Mountain View, CA. 
Requires 5+ years experience with Python and distributed systems. 
Salary range $180k-$250k."
Output:
- Title: Senior Software Engineer
- Company: Google
- Location: Mountain View, CA
- Experience: 5+ years
- Skills: Python, distributed systems
- Salary: $180k-$250k

Example 2:
Input: "Data Analyst role at Startup XYZ, remote position. 
Looking for someone with SQL expertise and 2-3 years in analytics."
Output:
- Title: Data Analyst
- Company: Startup XYZ
- Location: Remote
- Experience: 2-3 years
- Skills: SQL, analytics
- Salary: Not specified

Now extract:
Input: "AWS is hiring a Machine Learning Engineer in Seattle. 
Must have PhD or 7+ years ML experience. Strong TensorFlow and PyTorch skills required.
Competitive compensation with equity."
Output:
"""

print("Few-shot extraction:")
print(chat(few_shot_extraction, temperature=0))

---

## 5. Chain-of-Thought (CoT) Prompting

**Chain-of-thought** prompting asks the model to **show its reasoning step by step**, which significantly improves performance on complex tasks.

### Why It Works
- Forces the model to break down complex problems
- Reduces errors in multi-step reasoning
- Makes debugging easier (you can see where reasoning went wrong)

### Techniques
1. **Explicit CoT**: "Think step by step"
2. **Few-shot CoT**: Show examples with reasoning
3. **Self-Consistency**: Generate multiple chains, pick majority answer

In [ ]:
# Without Chain-of-Thought (often fails on complex math)
no_cot = """
A store sells apples for $0.50 each. If you buy 5 or more, you get a 20% discount.
How much would 8 apples cost?
"""

# With Chain-of-Thought
with_cot = """
A store sells apples for $0.50 each. If you buy 5 or more, you get a 20% discount.
How much would 8 apples cost?

Think through this step by step:
1. Calculate the base price
2. Check if discount applies
3. Apply discount if applicable
4. State the final answer
"""

print("With Chain-of-Thought:")
print(chat(with_cot, temperature=0))

In [ ]:
# Few-shot Chain-of-Thought for word problems
few_shot_cot = """
Solve the following problems by thinking step by step.

Problem: John has 3 times as many marbles as Sarah. Sarah has 7 marbles. 
How many marbles does John have?

Reasoning:
- Sarah has 7 marbles
- John has 3 times Sarah's amount
- John's marbles = 3 × 7 = 21
Answer: John has 21 marbles

---

Problem: A train travels at 60 mph. It needs to cover 150 miles. 
If it has already traveled for 1.5 hours, how much longer will it take?

Reasoning:
- Distance already covered = 60 mph × 1.5 hours = 90 miles
- Remaining distance = 150 - 90 = 60 miles
- Time needed for remaining = 60 miles ÷ 60 mph = 1 hour
Answer: 1 more hour

---

Problem: A rectangle's length is twice its width. If the perimeter is 36 cm, 
what are the dimensions?

Reasoning:
"""

print("Few-shot CoT:")
print(chat(few_shot_cot, temperature=0))

---

## 6. Structured Output Prompting

When you need outputs in specific formats (JSON, XML, tables), explicitly request the structure.

In [ ]:
# JSON Output
json_prompt = """
Extract information from this text and return valid JSON:

"Meeting scheduled for December 20th at 2:30 PM in Conference Room B. 
Attendees: John Smith, Sarah Johnson, Mike Chen. 
Topic: Q4 Budget Review"

Return ONLY valid JSON with keys: date, time, location, attendees (array), topic
"""

result = chat(json_prompt, temperature=0)
print("JSON Output:")
print(result)

# Verify it's valid JSON
import json
try:
    parsed = json.loads(result)
    print("\n✅ Valid JSON!")
except:
    print("\n⚠️ Not valid JSON")

In [ ]:
# Markdown Table Output
table_prompt = """
Compare Python, JavaScript, and Go programming languages.

Create a markdown table with columns:
- Language
- Primary Use Case
- Typing System
- Learning Curve (Easy/Medium/Hard)

Return ONLY the markdown table.
"""

print("Table Output:")
print(chat(table_prompt, temperature=0))

---

## 7. Role/Persona Prompting

Assigning a **role or persona** can dramatically change the style and expertise level of responses.

In [ ]:
# Same question, different personas
question = "What should I consider when designing a database schema?"

personas = {
    "Beginner-friendly teacher": 
        """You are a patient computer science teacher explaining to a first-year student. 
        Use simple analogies and avoid jargon.""",
    
    "Senior architect": 
        """You are a senior database architect at a Fortune 500 company. 
        Focus on enterprise-scale considerations and best practices.""",
    
    "Startup CTO":
        """You are a startup CTO who values speed and iteration. 
        Focus on practical trade-offs and avoiding over-engineering."""
}

for persona_name, persona_desc in personas.items():
    prompt = f"{persona_desc}\n\nQuestion: {question}\n\nAnswer (keep it concise, 3-4 sentences):"
    print(f"\n{'='*60}")
    print(f"🎭 Persona: {persona_name}")
    print(f"{'='*60}")
    print(chat(prompt, temperature=0.7))

---

## 8. Advanced Techniques

### Self-Refinement
Ask the model to critique and improve its own output.

In [ ]:
# Self-refinement prompting
self_refine = """
Task: Write a professional email declining a meeting invitation.

Step 1: Write a first draft.
Step 2: Critique your draft - what could be improved?
Step 3: Write an improved final version based on your critique.

Context: You're declining a meeting because of a schedule conflict, 
but want to suggest an alternative time.
"""

print("Self-refinement output:")
print(chat(self_refine, temperature=0.7))

### Constraint-Based Prompting
Explicitly list what the model should and shouldn't do.

In [ ]:
# Constraint-based prompting
constrained = """
Write a product description for a new smartphone.

MUST:
- Be exactly 3 sentences
- Include one specific technical spec
- End with a call-to-action

MUST NOT:
- Use superlatives (best, greatest, etc.)
- Mention competitors
- Use exclamation marks

Product: Galaxy X20 with 200MP camera
"""

print("Constrained output:")
print(chat(constrained, temperature=0.7))

---

## 9. Common Prompting Mistakes

### ❌ Mistakes to Avoid

| Mistake | Problem | Better Approach |
|---------|---------|----------------|
| Too vague | "Help me with my code" | "Fix the TypeError in this Python function..." |
| Too long | 2000-word context | Focus on relevant info only |
| No format specified | Output is unpredictable | "Return as JSON with keys..." |
| Contradictory instructions | Confuses the model | Review for consistency |
| Assuming knowledge | "Use the standard approach" | Be explicit about what approach |

### ✅ Debugging Prompts

When outputs aren't what you expect:
1. **Simplify**: Remove complexity, test core instruction
2. **Add examples**: Show what you want
3. **Be more explicit**: Don't assume the model understands
4. **Check temperature**: Lower for consistency, higher for creativity
5. **Split complex tasks**: Chain multiple simpler prompts

---

## 📝 Student Exercises

### Exercise 1: Classification System
Create a few-shot prompt that classifies customer support tickets into categories: BILLING, TECHNICAL, GENERAL, URGENT.

In [ ]:
# Exercise 1: Create your classification prompt
classification_prompt = """
# TODO: Add your few-shot examples here
# Then classify this ticket:

"My account was charged twice for the same order. Please refund the extra charge."
"""

# print(chat(classification_prompt, temperature=0))

### Exercise 2: Chain-of-Thought for Code Review
Create a CoT prompt that analyzes code for bugs, explaining the reasoning.

In [ ]:
# Exercise 2: Create your code review prompt with CoT
code_to_review = """
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)
"""

review_prompt = f"""
# TODO: Create a prompt that:
# 1. Asks for step-by-step analysis
# 2. Identifies potential bugs
# 3. Suggests improvements

Code to review:
{code_to_review}
"""

# print(chat(review_prompt, temperature=0))

### Exercise 3: Structured Data Extraction
Create a prompt that extracts structured data from unstructured text.

In [ ]:
# Exercise 3: Extract structured data
text_to_extract = """
Dr. Sarah Chen joined Microsoft's AI Research division in Seattle 
as a Principal Researcher in March 2023. She previously worked at 
Stanford University for 8 years and has published over 50 papers 
on natural language processing.
"""

extraction_prompt = f"""
# TODO: Create a prompt that extracts:
# - Name, Title, Company, Location, Start Date
# - Previous employer, Years there
# - Research area, Publications count
# Return as JSON

Text:
{text_to_extract}
"""

# print(chat(extraction_prompt, temperature=0))

---

## 🎯 Key Takeaways

1. **Good prompts are specific, structured, and include format requirements**
2. **Zero-shot** works for standard tasks; **few-shot** teaches custom patterns
3. **Chain-of-thought** dramatically improves reasoning on complex tasks
4. **Personas/roles** change expertise level and communication style
5. **Iterate and debug** prompts like you would debug code

### Prompt Engineering Cheat Sheet

| Technique | Use When |
|-----------|----------|
| Zero-shot | Standard tasks, quick tests |
| Few-shot | Custom formats, domain-specific |
| Chain-of-thought | Math, logic, multi-step reasoning |
| Role/Persona | Need specific expertise or tone |
| Structured output | Need JSON, tables, specific formats |
| Self-refinement | Quality is critical, time permits |

---

### Continue with:
- **02_function_calling.ipynb** - Connect LLMs to external tools
- **03_react_agent.ipynb** - Build reasoning agents

### Next Module: Fine-Tuning Pre-trained Models →